# 🧠 Notebook 05: VL-JEPA Architecture: Spatiotemporal Predictor, Cross-Modal Alignment & Joint Embedding
### Capstone: VL-JEPA vs. CLIP for CCTV Surveillance Event Retrieval, Event Summarization & Hallucination Analysis
---
**Objective:** Implement and train the **VL-JEPA (Vision-Language Joint Embedding Predictive Architecture)**:

#### Core Distinction (Why it is VL-JEPA, NOT V-JEPA):
- **V-JEPA (Visual JEPA):** Purely unimodal visual encoder trained by predicting masked spatiotemporal video tokens. It produces rich motion features but has **no language interface**.
- **VL-JEPA (Vision-Language JEPA):** Bridges spatiotemporal video representations with text supervision via a **trainable Spatiotemporal Cross-Modal Predictor** ($30.3\text{M}$ parameters) trained via Bidirectional InfoNCE Contrastive Loss.

#### Mathematical Formulation:
$$\mathcal{L}_{\text{InfoNCE}} = -\frac{1}{2} \left[ \sum_{i} \log \frac{\exp(z_{v,i} \cdot z_{t,i} / \tau)}{\sum_{j} \exp(z_{v,i} \cdot z_{t,j} / \tau)} + \sum_{i} \log \frac{\exp(z_{t,i} \cdot z_{v,i} / \tau)}{\sum_{j} \exp(z_{t,i} \cdot z_{v,j} / \tau)} \right]$$


## Step 1: Model Architecture Definition (30.3M Parameters)
Define the PyTorch `SpatiotemporalCrossModalPredictor` with 4-layer Transformer Encoder, 12 attention heads, and $d=768$.


In [1]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

sys.path.append(str(Path.cwd().parent))
import notebook_utils as nu

device = "cuda" if torch.cuda.is_available() else "cpu"

# Reset random seeds
torch.manual_seed(42)
np.random.seed(42)

model = nu.SpatiotemporalCrossModalPredictor(
    visual_dim=512,
    text_dim=512,
    shared_dim=768,
    num_layers=4,
    nhead=12,
    dim_feedforward=3072
).to(device)

criterion = nu.BiDirectionalInfoNCELoss(initial_temp=0.07).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"=== VL-JEPA Cross-Modal Alignment Predictor Initialized ===")
print(f"Trainable Parameters: {total_params:,} (~30.3M)")
print(f"Architecture:          4-layer Transformer Encoder (12 heads, d_model=768, d_ff=3072)")
print(f"Device:                {device}")


=== VL-JEPA Cross-Modal Alignment Predictor Initialized ===
Trainable Parameters: 30,326,785 (~30.3M)
Architecture:          4-layer Transformer Encoder (12 heads, d_model=768, d_ff=3072)
Device:                cuda


## Step 2: Sample-Specific Text Supervision Generation
Generate rich sample-specific text prompts incorporating viewpoint, actions, and anomaly metadata.


In [2]:
_, metadata = nu.load_all_datasets_data()
sample_prompts = [nu.generate_sample_prompt(m) for m in metadata]

print(f"Generated {len(sample_prompts):,} sample-specific text descriptions.")
print("\n--- Examples across Datasets ---")
print(f"RLVS:       {sample_prompts[0]}")
print(f"UCF-ARG:    {sample_prompts[2000]}")
print(f"TinyVIRAT:  {sample_prompts[3500]}")
print(f"UCF-Crime:  {sample_prompts[4500]}")


[Dataset Loader] Loaded RLVS: 2000 clips, dim 512
[Dataset Loader] Loaded UCF_ARG: 1410 clips, dim 512
[Dataset Loader] Loaded TINYVIRAT: 743 clips, dim 512
[Dataset Loader] Loaded UCF_CRIME: 1000 clips, dim 512
Generated 5,153 sample-specific text descriptions.

--- Examples across Datasets ---
RLVS:       a CCTV surveillance camera recording violent street fighting, punching, physical brawl, or assault, surveillance footage sequence V_1
UCF-ARG:    two people boxing, throwing punches, or fighting in an outdoor area, captured from an outdoor aerial surveillance angle of person01
TinyVIRAT:  a person standing still or waiting in place, fixed surveillance camera viewpoint
UCF-Crime:  a burglar breaking into a building, home burglary, forced door entry, or house robbery, incident clip Burglary098_x264.mp4


## Step 3: Inspect Trained Checkpoint and Projection Matrix
Load the trained model weights and verify the aligned latent representations.


In [3]:
checkpoint_path = Path.cwd().parent / "outputs" / "checkpoints" / "vljepa_predictor_trained.pt"
if checkpoint_path.exists():
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    print(f"[OK] Successfully loaded trained weights from: {checkpoint_path.name}")
else:
    print("[INFO] Checkpoint not found on disk; model initialized with fresh seed weights.")

# Test forward projection
dummy_v = torch.randn(2, 512).to(device)
dummy_t = torch.randn(2, 512).to(device)
with torch.no_grad():
    zv, zt = model(dummy_v, dummy_t)
print(f"Visual projected shape: {zv.shape} (L2 norm: {zv.norm(dim=-1).cpu().numpy()})")
print(f"Text projected shape:   {zt.shape} (L2 norm: {zt.norm(dim=-1).cpu().numpy()})")


[OK] Successfully loaded trained weights from: vljepa_predictor_trained.pt
Visual projected shape: torch.Size([2, 768]) (L2 norm: [1. 1.])
Text projected shape:   torch.Size([2, 768]) (L2 norm: [1. 1.])
